# 01 — Data Preparation

**Fingo Income Predictor** | Tim CC26-PSU217 | DS2 Clarisya Adeline

Input:
- `data/raw/form_responses.csv`
- BPS files di `data/raw/`

Output:
- `data/processed/survey_clean.csv`

Catatan:
- `timestamp` dan `timestamp_parsed` **tidak boleh di-drop** pada notebook ini karena dipakai oleh `02_Temporal_Mapping.ipynb`.
- Urutan kronologis income untuk model adalah `income_w4 → income_w3 → income_w2 → income_w1`.

In [1]:
# GIT PULL — Sinkronisasi terbaru dari remote sebelum mulai
import os, shutil, subprocess

try:
    from google.colab import userdata
except Exception:
    userdata = None

os.chdir("/content")

GITHUB_USERNAME = "ClarisyaA"
REPO_NAME       = "fingo-income-analysis"
BRANCH_NAME     = "feat/income-predictor-final"
LOCAL_DIR       = f"/content/{REPO_NAME}"
FRESH_CLONE     = False  # Set True hanya kalau mau clone ulang dari nol

def get_remote_url():
    try:
        token = userdata.get("GITHUB_TOKEN") if userdata else os.environ.get("GITHUB_TOKEN", "")
        if token:
            return f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", token
    except Exception:
        pass
    return f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", None

remote_url, token = get_remote_url()

def mask_cmd(cmd):
    return cmd.replace(token, "***TOKEN***") if token else cmd

def run_cmd(cmd, check=True, cwd="/content"):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    print(f"$ {mask_cmd(cmd)}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {mask_cmd(cmd)}")
    return r

def remote_branch_exists():
    r = run_cmd(f"git ls-remote --heads {remote_url} {BRANCH_NAME}", check=False)
    return r.stdout.strip() != ""

branch_exists = remote_branch_exists()

if FRESH_CLONE and os.path.exists(LOCAL_DIR):
    os.chdir("/content")
    shutil.rmtree(LOCAL_DIR)

if not os.path.exists(LOCAL_DIR):
    if branch_exists:
        run_cmd(f"git clone -b {BRANCH_NAME} {remote_url} {LOCAL_DIR}")
    else:
        run_cmd(f"git clone {remote_url} {LOCAL_DIR}")
        run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)
else:
    run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
    run_cmd("git fetch origin", cwd=LOCAL_DIR)

    if branch_exists:
        local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
        if local_b:
            run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
        else:
            run_cmd(f"git checkout -b {BRANCH_NAME} origin/{BRANCH_NAME}", cwd=LOCAL_DIR)
        run_cmd(f"git pull --rebase origin {BRANCH_NAME}", cwd=LOCAL_DIR)
    else:
        current = run_cmd("git branch --show-current", check=False, cwd=LOCAL_DIR).stdout.strip()
        if current != BRANCH_NAME:
            local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
            if local_b:
                run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
            else:
                run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)

os.chdir(LOCAL_DIR)
run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
print("\n✓ Repo siap digunakan")
print(f"✓ Working directory: {os.getcwd()}")
run_cmd("git branch --show-current", cwd=LOCAL_DIR)
run_cmd("git status --short", check=False, cwd=LOCAL_DIR)

$ git ls-remote --heads https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git feat/income-predictor-final
$ git clone https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git /content/fingo-income-analysis
Cloning into '/content/fingo-income-analysis'...
$ git checkout -b feat/income-predictor-final
Switched to a new branch 'feat/income-predictor-final'
$ git remote set-url origin https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git

✓ Repo siap digunakan
✓ Working directory: /content/fingo-income-analysis
$ git branch --show-current
feat/income-predictor-final
$ git status --short


CompletedProcess(args='git status --short', returncode=0, stdout='', stderr='')

In [2]:
# CELL 01.2 — Buat struktur folder
import os
LOCAL_DIR = "/content/fingo-income-analysis"
os.chdir(LOCAL_DIR)

folders = [
    "data/raw", "data/processed", "data/synthetic",
    "outputs/charts", "outputs/dashboard", "outputs/model_results",
    "outputs/preprocessors", "outputs/model_contract", "outputs/reports",
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("✓ Struktur folder siap")
print(f"✓ CWD: {os.getcwd()}")

✓ Struktur folder siap
✓ CWD: /content/fingo-income-analysis


In [3]:
# CELL 01.3 — Install & import library
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "xgboost", "scikit-learn", "scipy", "--quiet"])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, json, pickle, os, re
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
np.random.seed(42)

IDR_FMT = mticker.FuncFormatter(lambda x, _: f"Rp {x/1e6:.1f}jt" if x >= 1e6 else f"Rp {x/1e3:.0f}rb")

def fmt_idr(val):
    if pd.isna(val): return "NaN"
    if val >= 1_000_000: return f"Rp {val/1_000_000:.2f}jt"
    return f"Rp {val/1_000:.0f}rb"

SIMULATION_YEAR   = 2026
N_SYNTHETIC_USERS = 3000
N_WEEKS           = 52

plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid", palette="Set2")
print("✓ Library siap")

✓ Library siap


In [4]:
# CELL 01.4 — Helper functions
import os, warnings

def ensure_dir(path):
    d = os.path.dirname(path) if "." in os.path.basename(path) else path
    if d: os.makedirs(d, exist_ok=True)

def safe_read_csv(path, **kwargs):
    for sep in [",", ";", "\t"]:
        try:
            df = pd.read_csv(path, sep=sep, **kwargs)
            if len(df.columns) > 1: return df
        except Exception:
            continue
    warnings.warn(f"[WARN] Gagal membaca {path}")
    return pd.DataFrame()

def find_file_by_keywords(folder, keywords):
    if not os.path.isdir(folder): return None
    for fname in os.listdir(folder):
        if all(kw.lower() in fname.lower() for kw in keywords):
            return os.path.join(folder, fname)
    return None

def safe_to_csv(df, path, **kwargs):
    ensure_dir(path)
    df.to_csv(path, index=kwargs.pop("index", False), **kwargs)
    return True

def safe_savefig(path, **kwargs):
    ensure_dir(path)
    plt.savefig(path, dpi=kwargs.pop("dpi", 150), bbox_inches="tight", **kwargs)

def get_week_of_month(date):
    if pd.isna(date): return np.nan
    return int((date.day - 1) // 7 + 1)

print("✓ Helper functions siap")

✓ Helper functions siap


## Load Raw Data

In [5]:
# CELL 01.5 — Load data survei utama
SURVEY_CANDIDATES = [
    "data/raw/form_responses.csv",
    "form_responses.csv",
]

df_raw = None
for candidate in SURVEY_CANDIDATES:
    if os.path.exists(candidate):
        try:
            df_raw = pd.read_csv(candidate)
            print(f"✓ Data survei dimuat dari: {candidate}")
            break
        except Exception as e:
            print(f"  Gagal baca {candidate}: {e}")

if df_raw is None:
    if os.path.isdir("data/raw"):
        for f in os.listdir("data/raw"):
            if "form" in f.lower() and f.endswith(".csv"):
                df_raw = pd.read_csv(os.path.join("data/raw", f))
                print(f"✓ Data survei dimuat dari: data/raw/{f}")
                break

if df_raw is None:
    raise FileNotFoundError("File survei tidak ditemukan. Upload form_responses.csv ke data/raw/")

print(f"✓ Shape: {df_raw.shape[0]} baris × {df_raw.shape[1]} kolom")
if df_raw.shape[0] != 384:
    print(f"[WARN] Expected 384 responden, got {df_raw.shape[0]}")
else:
    print("✓ 384 responden terverifikasi")

✓ Data survei dimuat dari: data/raw/form_responses.csv
✓ Shape: 384 baris × 20 kolom
✓ 384 responden terverifikasi


In [6]:
# CELL 01.6 — Load data BPS
BPS_FILE_KEYWORDS = {
    "bps_bebas_2025":    ["Bebas", "2025"],
    "bps_bebas_2024":    ["Bebas", "2024"],
    "bps_informal_2025": ["Informal", "2025"],
    "bps_informal_2023": ["Informal", "2023"],
}

bps_dfs = {}
raw_dir = "data/raw"

for key, keywords in BPS_FILE_KEYWORDS.items():
    found = find_file_by_keywords(raw_dir, keywords)
    if found:
        try:
            bps_dfs[key] = safe_read_csv(found)
            print(f"  {key}: {os.path.basename(found)} ({bps_dfs[key].shape})")
        except Exception as e:
            print(f"  {key}: gagal membaca — {e}")
    else:
        print(f"  {key}: file tidak ditemukan (keywords: {keywords})")

print(f"\n✓ BPS file berhasil dimuat: {len(bps_dfs)}/{len(BPS_FILE_KEYWORDS)}")

  bps_bebas_2025: Rata-Rata Pendapatan Bersih Sebulan Pekerja Bebas Menurut Provinsi dan Lapangan Pekerjaan Utama, 2025.csv ((43, 13))
  bps_bebas_2024: Rata-Rata_Pendapatan_Bersih_Sebulan_Pekerja_Bebas_Menurut_Provinsi_dan_Lapangan_Pekerjaan_Utama_2024.csv ((39, 9))
  bps_informal_2025: Rata-Rata Pendapatan Bersih Sebulan Pekerja Informal Menurut Provinsi dan Lapangan Pekerjaan Utama (rupiah), 2025.csv ((44, 5))
  bps_informal_2023: Rata-rata Pendapatan Bersih Sebulan Pekerja Informal Menurut Provinsi dan Lapangan Pekerjaan Utama (rupiah), 2023.csv ((46, 5))

✓ BPS file berhasil dimuat: 4/4


## Form Response Mapping

Mapping berdasarkan posisi kolom agar robust terhadap perubahan nama form.

| Col | Kolom Clean | Keterangan |
|---|---|---|
| 0 | timestamp | timestamp responden |
| 10 | income_w1 | Pendapatan minggu lalu / terbaru |
| 11 | income_w2 | Dua minggu lalu |
| 12 | income_w3 | Tiga minggu lalu |
| 13 | income_w4 | Empat minggu lalu / terlama |

Urutan kronologis: `income_w4 → income_w3 → income_w2 → income_w1`

In [7]:
# CELL 01.7 — Form column mapping
cols = df_raw.columns.tolist()

if len(cols) < 20:
    raise ValueError(f"Kolom form hanya {len(cols)}, minimal 20. Periksa file survei.")

FORM_RENAME_MAP = {
    cols[0]:  "timestamp",
    cols[1]:  "consent",
    cols[2]:  "usia",
    cols[3]:  "domisili",
    cols[4]:  "pekerjaan",
    cols[5]:  "sumber_pekerjaan",
    cols[6]:  "status_income",
    cols[7]:  "lama_kerja_bulan",
    cols[8]:  "hari_kerja_per_minggu",
    cols[9]:  "jam_kerja_per_hari",
    cols[10]: "income_w1",
    cols[11]: "income_w2",
    cols[12]: "income_w3",
    cols[13]: "income_w4",
    cols[14]: "peak_week",
    cols[15]: "waktu_ramai",
    cols[16]: "faktor_fluktuasi",
    cols[17]: "app_helpful_score",
    cols[18]: "fitur_dibutuhkan",
    cols[19]: "kontak_gopay",
}

mapping_export = {str(i): {"original": k, "clean": v}
                  for i, (k, v) in enumerate(FORM_RENAME_MAP.items())}
ensure_dir("outputs/reports/form_column_mapping.json")
with open("outputs/reports/form_column_mapping.json", "w", encoding="utf-8") as f:
    json.dump(mapping_export, f, indent=2, ensure_ascii=False)

print("✓ Form column mapping:")
for i, (orig, clean) in enumerate(FORM_RENAME_MAP.items()):
    orig_short = orig[:55] + "..." if len(orig) > 55 else orig
    print(f"  col[{i:2d}] → {clean:25s} ← {orig_short}")

print("\n✓ Urutan kronologis income:")
print("  income_w4 (4mgg lalu/TERLAMA) → income_w3 → income_w2 → income_w1 (TERBARU)")

✓ Form column mapping:
  col[ 0] → timestamp                 ← Timestamp
  col[ 1] → consent                   ← Apakah Anda bersedia mengisi survei ini?

Cuman sedikit...
  col[ 2] → usia                      ← Berapa usia Anda? (tulis angka, contoh: 22)
  col[ 3] → domisili                  ← Domisili Anda saat ini
  col[ 4] → pekerjaan                 ← Pekerjaan utama Anda saat ini?

Kalau punya lebih dari ...
  col[ 5] → sumber_pekerjaan          ← Biasanya Anda mendapatkan pekerjaan/pesanan dari mana? ...
  col[ 6] → status_income             ← Apakah pekerjaan ini menjadi sumber penghasilan utama A...
  col[ 7] → lama_kerja_bulan          ← Sudah berapa lama Anda menjalankan pekerjaan ini? (tuli...
  col[ 8] → hari_kerja_per_minggu     ← Berapa hari dalam seminggu ada bekerja?
  col[ 9] → jam_kerja_per_hari        ← Berapa jam Anda bekerja dalam sehari? (tulis angka, con...
  col[10] → income_w1                 ← Berapa penghasilan Anda minggu lalu? (tulis dalam Rupia...
  col[1

## Data Cleaning

In [8]:
# CELL 01.8 — Rename + Drop PII
# timestamp DIPERTAHANKAN — diperlukan untuk temporal mapping di notebook 02
df = df_raw.rename(columns=FORM_RENAME_MAP).copy()

pii_drop = ["consent", "kontak_gopay"]
df.drop(columns=[c for c in pii_drop if c in df.columns], inplace=True)

print("✓ Rename kolom selesai")
print(f"  Baris   : {df.shape[0]}")
print(f"  Kolom   : {df.shape[1]}")
print(f"  PII drop: {pii_drop}")
print("  timestamp DIPERTAHANKAN untuk temporal mapping (02_Temporal_Mapping.ipynb)")

✓ Rename kolom selesai
  Baris   : 384
  Kolom   : 18
  PII drop: ['consent', 'kontak_gopay']
  timestamp DIPERTAHANKAN untuk temporal mapping (02_Temporal_Mapping.ipynb)


In [9]:
# CELL 01.9 — Drop duplikasi + respondent_id
n_before = len(df)
df.drop_duplicates(inplace=True)
n_after = len(df)
print(f"✓ Drop duplikasi: {n_before - n_after} baris dihapus → tersisa {n_after}")
df = df.reset_index(drop=True)
df["respondent_id"] = ["R" + str(i).zfill(4) for i in range(len(df))]
print(f"✓ respondent_id dibuat: R0000 – R{len(df)-1:04d}")

✓ Drop duplikasi: 0 baris dihapus → tersisa 384
✓ respondent_id dibuat: R0000 – R0383


In [10]:
# CELL 01.10 — Parse timestamp (SIMPAN, jangan drop)
if "timestamp" in df.columns:
    df["timestamp_parsed"] = pd.to_datetime(df["timestamp"], errors="coerce")
    n_valid = df["timestamp_parsed"].notna().sum()
    print(f"✓ timestamp_parsed: {n_valid}/{len(df)} valid")
    print(f"  Range: {df['timestamp_parsed'].min()} — {df['timestamp_parsed'].max()}")
    df["survey_date"] = df["timestamp_parsed"].dt.date
else:
    print("timestamp tidak ditemukan — temporal mapping tidak bisa dilakukan")
    df["timestamp_parsed"] = pd.NaT
    df["survey_date"] = None

✓ timestamp_parsed: 384/384 valid
  Range: 2026-05-16 08:25:17 — 2026-05-26 22:27:26


In [11]:
# CELL 01.11 — Convert numerik + clip nilai tidak realistis
income_cols = ["income_w1", "income_w2", "income_w3", "income_w4"]
for col in income_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0)

df["usia"] = pd.to_numeric(df["usia"], errors="coerce").fillna(25).clip(17, 65)
df["hari_kerja_per_minggu"] = pd.to_numeric(df["hari_kerja_per_minggu"], errors="coerce").fillna(5).clip(1, 7)
df["jam_kerja_per_hari"] = pd.to_numeric(df["jam_kerja_per_hari"], errors="coerce").fillna(8).clip(1, 16)
df["lama_kerja_bulan"] = pd.to_numeric(df["lama_kerja_bulan"], errors="coerce").fillna(6).clip(1, None)
df["status_income"] = pd.to_numeric(df["status_income"], errors="coerce").fillna(1).clip(0, 2)
df["peak_week"] = pd.to_numeric(df["peak_week"], errors="coerce").fillna(0).clip(0, 4)
df["app_helpful_score"] = pd.to_numeric(df["app_helpful_score"], errors="coerce").fillna(3).clip(1, 5)

print("✓ Konversi numerik selesai")
for col in income_cols:
    print(f"  {col}: min={df[col].min():,.0f}, max={df[col].max():,.0f}, mean={df[col].mean():,.0f}")

✓ Konversi numerik selesai
  income_w1: min=0, max=1,955,000, mean=439,378
  income_w2: min=0, max=2,245,000, mean=455,836
  income_w3: min=0, max=2,375,000, mean=443,237
  income_w4: min=0, max=2,510,000, mean=447,318


In [12]:
# CELL 01.12 — Standardisasi kategori
GIG_MAP = {
    "Ojek online / driver aplikasi":              "ojek_online",
    "Kurir / pengantar barang atau makanan":      "kurir",
    "Jualan online / reseller / toko online":     "jualan_online",
    "Freelance desain / editing / ilustrasi":     "freelance_desain",
    "Freelance IT / website / programming / data":"freelance_it",
    "Content creator / admin media sosial":       "content_creator",
    "Tutor / guru les / pengajar lepas":          "tutor",
    "Pekerja harian / event / part-time":         "pekerja_harian",
}

DOMISILI_MAP = {
    "Jabodetabek (Jakarta, Bogor, Depok, Tangerang, Bekasi)": "jabodetabek",
    "Bandung":                   "bandung",
    "Jawa Barat lainnya":        "jabar_lainnya",
    "Jawa Tengah / Yogyakarta":  "jateng_yogya",
    "Jawa Timur":                "jatim",
    "Sumatera":                  "sumatera",
    "Kalimantan":                "kalimantan",
    "Sulawesi":                  "sulawesi",
    "Bali / Nusa Tenggara":      "bali_ntt",
}

df["gig_type"] = df["pekerjaan"].map(GIG_MAP).fillna("lainnya")
df["domisili_code"] = df["domisili"].map(DOMISILI_MAP).fillna("lainnya")

ORDERED_GIG_TYPES = ["ojek_online", "kurir", "jualan_online", "freelance_desain",
                     "freelance_it", "content_creator", "tutor", "pekerja_harian"]

print("✓ GIG_MAP applied:")
print(df["gig_type"].value_counts().to_string())
print("\n✓ DOMISILI_MAP applied:")
print(df["domisili_code"].value_counts().to_string())

✓ GIG_MAP applied:
gig_type
pekerja_harian      66
jualan_online       63
tutor               61
freelance_desain    58
freelance_it        46
content_creator     34
ojek_online         29
kurir               27

✓ DOMISILI_MAP applied:
domisili_code
jabodetabek      116
bandung           82
jateng_yogya      43
jabar_lainnya     40
jatim             33
sumatera          26
sulawesi          19
bali_ntt          13
kalimantan        12


In [13]:
# CELL 01.13 — Multi-hot encoding kolom multi-select
SRC_OPTIONS = {
    "Gojek": "src_gojek",
    "Grab": "src_grab",
    "ShopeeFood": "src_shopeefood",
    "Maxim": "src_maxim",
    "Shopee / Tokopedia / TikTok Shop": "src_marketplace",
    "Instagram / TikTok / YouTube": "src_sosmed",
    "WhatsApp / kenalan / pelanggan langsung": "src_wa_langsung",
    "Website freelance seperti Fiverr, Upwork, Projects.co.id": "src_freelance_web",
}
RAMAI_OPTIONS = {
    "Awal bulan": "ramai_awal_bulan",
    "Akhir bulan / tanggal gajian": "ramai_akhir_bulan",
    "Akhir pekan / Sabtu-Minggu": "ramai_weekend",
    "Ramadan / menjelang Lebaran": "ramai_ramadan",
    "Natal / Tahun Baru": "ramai_natal",
    "Harbolnas / flash sale online / tanggal kembar": "ramai_harbolnas",
    "Saat ada promo dari aplikasi": "ramai_promo_app",
    "Saat cuaca bagus": "ramai_cuaca_bagus",
}
FLUK_OPTIONS = {
    "Jumlah pesanan atau project yang masuk": "fluk_jumlah_pesanan",
    "Jam saya bekerja": "fluk_jam_kerja",
    "Cuaca": "fluk_cuaca",
    "Akhir pekan atau hari libur": "fluk_weekend",
    "Ramadan / Lebaran": "fluk_ramadan",
    "Tanggal gajian / akhir bulan": "fluk_gajian",
    "Ada atau tidaknya promo dari aplikasi": "fluk_promo_app",
    "Rating atau ulasan pelanggan": "fluk_rating",
    "Kondisi fisik / capek / sakit": "fluk_kondisi_fisik",
}
FITUR_OPTIONS = {
    "Perkiraan penghasilan minggu depan": "fitur_perkiraan",
    "Pengingat kalau penghasilan sedang turun": "fitur_pengingat_turun",
    "Saran batas pengeluaran minggu ini": "fitur_saran_pengeluaran",
    "Saran menabung": "fitur_saran_nabung",
    "Ringkasan pemasukan dan pengeluaran": "fitur_ringkasan",
    "Tips mengatur uang buat pekerja seperti saya": "fitur_tips_uang",
}

def apply_multihot(df, col, option_map):
    if col not in df.columns:
        for colname in option_map.values():
            df[colname] = 0
        return df
    for option, colname in option_map.items():
        df[colname] = df[col].astype(str).str.contains(re.escape(option), na=False).astype(int)
    return df

df = apply_multihot(df, "sumber_pekerjaan", SRC_OPTIONS)
df = apply_multihot(df, "waktu_ramai", RAMAI_OPTIONS)
df = apply_multihot(df, "faktor_fluktuasi", FLUK_OPTIONS)
df = apply_multihot(df, "fitur_dibutuhkan", FITUR_OPTIONS)

df.drop(columns=["sumber_pekerjaan", "waktu_ramai", "faktor_fluktuasi", "fitur_dibutuhkan",
                 "pekerjaan", "domisili"], errors="ignore", inplace=True)

print("✓ Multi-hot encoding selesai")

✓ Multi-hot encoding selesai


In [14]:
# CELL 01.14 — Feature engineering dasar
df["total_jam_seminggu"] = df["hari_kerja_per_minggu"] * df["jam_kerja_per_hari"]
df["experience_months_log"] = np.log1p(df["lama_kerja_bulan"])

income_cols_ordered = ["income_w4", "income_w3", "income_w2", "income_w1"]
valid_income = df[income_cols_ordered].replace(0, np.nan)
df["avg_weekly_income"] = valid_income.mean(axis=1).fillna(0)
df["monthly_income"] = df[income_cols_ordered].sum(axis=1)
df["income_std_4w"] = valid_income.std(axis=1).fillna(0)
df["income_cv_4w"] = np.where(
    df["avg_weekly_income"] > 0,
    df["income_std_4w"] / df["avg_weekly_income"], 0
)
df["income_range_4w"] = df[income_cols_ordered].max(axis=1) - df[income_cols_ordered].min(axis=1)

df["pref_payday"] = (df.get("ramai_akhir_bulan", 0) | df.get("ramai_awal_bulan", 0)).clip(0, 1)
df["pref_awal_bulan"] = df.get("ramai_awal_bulan", pd.Series(0, index=df.index))
df["pref_weekend"] = df.get("ramai_weekend", pd.Series(0, index=df.index))
df["pref_ramadan_lebaran"] = df.get("ramai_ramadan", pd.Series(0, index=df.index))
df["pref_natal_tahun_baru"] = df.get("ramai_natal", pd.Series(0, index=df.index))
df["pref_harbolnas"] = df.get("ramai_harbolnas", pd.Series(0, index=df.index))
df["pref_promo_aplikasi"] = df.get("ramai_promo_app", pd.Series(0, index=df.index))

print("✓ Feature engineering selesai")
print("  Income ordering: income_w4 (terlama) → income_w1 (terbaru)")
print(f"  avg_weekly_income: {df['avg_weekly_income'].describe().round(0).to_dict()}")

✓ Feature engineering selesai
  Income ordering: income_w4 (terlama) → income_w1 (terbaru)
  avg_weekly_income: {'count': 384.0, 'mean': 476047.0, 'std': 326363.0, 'min': 75000.0, '25%': 239583.0, '50%': 384000.0, '75%': 614062.0, 'max': 2271250.0}


In [15]:
# CELL 01.15 — Validasi missing value + data type
print("=== Validasi survey_clean ===")
mv = df.isnull().sum()
print("Missing values:")
print(mv[mv > 0].to_string() if mv.sum() > 0 else "  Tidak ada missing values pada kolom utama")
print(f"\nShape: {df.shape}")
print(f"Duplikasi: {df.duplicated().sum()}")
print("\nKolom yang ada:")
print(df.dtypes.to_string())

=== Validasi survey_clean ===
Missing values:
  Tidak ada missing values pada kolom utama

Shape: (384, 62)
Duplikasi: 0

Kolom yang ada:
timestamp                          object
usia                                int64
status_income                       int64
lama_kerja_bulan                    int64
hari_kerja_per_minggu               int64
jam_kerja_per_hari                  int64
income_w1                           int64
income_w2                           int64
income_w3                           int64
income_w4                           int64
peak_week                           int64
app_helpful_score                   int64
respondent_id                      object
timestamp_parsed           datetime64[ns]
survey_date                        object
gig_type                           object
domisili_code                      object
src_gojek                           int64
src_grab                            int64
src_shopeefood                      int64
src_maxim             

## Simpan Output

`timestamp` dan `timestamp_parsed` wajib tetap ada di `survey_clean.csv`.

In [16]:
# CELL 01.16 — Simpan survey_clean.csv
safe_to_csv(df, "data/processed/survey_clean.csv")
print("✓ Disimpan: data/processed/survey_clean.csv")
print(f"  Shape: {df.shape}")
print(f"  Kolom timestamp ada: {'timestamp' in df.columns and 'timestamp_parsed' in df.columns}")
print(f"  Kolom income_w1–w4: {all(f'income_w{i}' in df.columns for i in range(1,5))}")

✓ Disimpan: data/processed/survey_clean.csv
  Shape: (384, 62)
  Kolom timestamp ada: True
  Kolom income_w1–w4: True


In [17]:
# GIT PUSH — Commit dan push output notebook ini ke GitHub
import os, subprocess

LOCAL_DIR   = "/content/fingo-income-analysis"
BRANCH_NAME = "feat/income-predictor-final"
NOTEBOOK_NAME = "01_Data_Preparation.ipynb"

os.chdir(LOCAL_DIR)

def run_cmd(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {cmd}")
    return r

run_cmd('git config user.email "adelineclarisya@gmail.com"')
run_cmd('git config user.name "ClarisyaA"')

print("\n[1] Cek status")
run_cmd("git status --short", check=False)

print("\n[2] Add semua perubahan output")
run_cmd("git add data/ outputs/ notebooks/ *.ipynb", check=False)

print("\n[3] Commit")
commit_result = run_cmd(
    f'git commit -m "feat(DS2): output dari {NOTEBOOK_NAME}"',
    check=False
)
if commit_result.returncode != 0:
    print("[INFO] Tidak ada perubahan baru, skip commit.")

print("\n[4] Fetch remote terbaru")
run_cmd("git fetch origin")

print("\n[5] Rebase lalu push")
run_cmd(f"git pull --rebase origin {BRANCH_NAME}", check=False)
run_cmd(f"git push -u origin {BRANCH_NAME}")

print("\n✓ Push berhasil!")

$ git config user.email "adelineclarisya@gmail.com"
$ git config user.name "ClarisyaA"

[1] Cek status
$ git status --short
?? data/processed/survey_clean.csv

[2] Add semua perubahan output
$ git add data/ outputs/ notebooks/ *.ipynb

[3] Commit
$ git commit -m "feat(DS2): output dari 01_Data_Preparation.ipynb"
[feat/income-predictor-final b8b0613] feat(DS2): output dari 01_Data_Preparation.ipynb
 1 file changed, 385 insertions(+)
 create mode 100644 data/processed/survey_clean.csv

[4] Fetch remote terbaru
$ git fetch origin

[5] Rebase lalu push
$ git pull --rebase origin feat/income-predictor-final
fatal: couldn't find remote ref feat/income-predictor-final
$ git push -u origin feat/income-predictor-final
Branch 'feat/income-predictor-final' set up to track remote branch 'feat/income-predictor-final' from 'origin'.
remote: 
remote: Create a pull request for 'feat/income-predictor-final' on GitHub by visiting:        
remote:      https://github.com/ClarisyaA/fingo-income-analysis/p